



# Part D: Inspect the Sparse Representation



## 1. Cài đặt và tìm dataset


In [1]:
# Path xử lý đường dẫn; Counter dùng để đếm term; các module còn lại phục vụ JSON, regex và toán học.
from pathlib import Path
from collections import Counter
import json
import math
import re
import subprocess
import sys

# Tên dataset và URL repository dùng khi notebook chạy trên Google Colab.
DATA_FILENAME = 'c4-train.00000-of-01024-30K.json'
REPO_URL = 'https://github.com/NguyenDucThang-tb/NLP.git'

# Thử lần lượt các vị trí dataset trên local, Colab và thư mục hiện tại.
candidate_paths = [
    Path('/home/nguyenducthang/NLP/Lab1') / DATA_FILENAME,
    Path('/content/NLP/Lab1') / DATA_FILENAME,
    Path.cwd() / DATA_FILENAME,
    Path.cwd() / 'Lab1' / DATA_FILENAME,
]
# Lấy path đầu tiên tồn tại; nếu không có thì nhận giá trị None.
DATA_PATH = next((p for p in candidate_paths if p.is_file()), None)

# Nếu đang ở Colab mà chưa có dataset, clone repository rồi tìm file bên trong.
if DATA_PATH is None and Path('/content').exists():
    repo_dir = Path('/content/NLP')
    if not (repo_dir / '.git').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
    matches = list(repo_dir.rglob(DATA_FILENAME))
    DATA_PATH = matches[0] if matches else None

# Báo lỗi sớm để người dùng biết chính xác cần đặt dataset ở đâu.
if DATA_PATH is None:
    raise FileNotFoundError(
        f'Không tìm thấy {DATA_FILENAME}. Đặt dataset vào thư mục Lab1 hoặc /content/NLP/Lab1.'
    )

print('Dataset:', DATA_PATH)

Dataset: /content/NLP/Lab1/c4-train.00000-of-01024-30K.json


## 2. Dataset và tokenizer


In [2]:
# Regex lấy các chuỗi ký tự dạng word; lowercase được thực hiện trong tokenizer.
TOKEN_RE = re.compile(r'(?u)\b\w+\b')

# Chuyển text thành danh sách token chuẩn hóa.
def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

# Các biến thống kê chất lượng dữ liệu.
raw_count = 0
invalid_count = 0
empty_count = 0
documents = []

# Dataset là JSONL: đọc từng dòng thay vì load toàn bộ file một lần.
with DATA_PATH.open('r', encoding='utf-8') as file:
    for line in file:
        raw_count += 1
        # Mỗi dòng phải được parse độc lập vì một dòng hỏng không nên làm mất toàn bộ corpus.
        try:
            item = json.loads(line)
            text = item.get('text', '') if isinstance(item, dict) else ''
        except (json.JSONDecodeError, UnicodeDecodeError):
            invalid_count += 1
            continue
        # Bỏ document không có token hợp lệ.
        if not tokenize(text):
            empty_count += 1
            continue
        documents.append({'text': text, 'url': item.get('url', '')})

# Tách text và token lengths để các bước sau dùng thuận tiện hơn.
texts = [doc['text'] for doc in documents]
N = len(texts)
token_lists = [tokenize(text) for text in texts]
token_lengths = [len(tokens) for tokens in token_lists]

print(f'Raw documents = {raw_count:,}')
print(f'Valid documents = {N:,}')
print(f'Invalid JSON lines = {invalid_count:,}')
print(f'Empty documents = {empty_count:,}')
print(f'Average tokens/document = {sum(token_lengths) / N:.2f}')
print(f'Min tokens/document = {min(token_lengths)}')
print(f'Max tokens/document = {max(token_lengths)}')

Raw documents = 30,000
Valid documents = 30,000
Invalid JSON lines = 0
Empty documents = 0
Average tokens/document = 369.70
Min tokens/document = 5
Max tokens/document = 23445


## 3. Xây dựng Count, TF, IDF và TF-IDF sparse


In [3]:
# Vocabulary ánh xạ term -> column index; sorted giúp thứ tự cột luôn tái lập được.
vocabulary = {term: index for index, term in enumerate(sorted({t for tokens in token_lists for t in tokens}))}
V = len(vocabulary)

# Count representation: mỗi document là một Counter chỉ lưu term xuất hiện.
count_rows = [Counter(tokens) for tokens in token_lists]

# df(t) là số document chứa term t, không phải tổng số lần term xuất hiện.
df = Counter()
for row in count_rows:
    df.update(row.keys())

# IDF theo đúng công thức của đề: term càng hiếm thì IDF càng lớn.
idf = {term: math.log(N / df[term]) for term in vocabulary}

# Mỗi row TF-IDF là dictionary sparse thay vì vector dense kích thước V.
tfidf_rows = []
for tokens, counts in zip(token_lists, count_rows):
    # length là tổng số token của document, dùng để chuẩn hóa TF.
    length = len(tokens)
    row = {}
    for term, count in counts.items():
        # TF = count/length; TF-IDF = TF * IDF.
        value = (count / length) * idf[term]
        if value != 0.0:
            row[term] = value
    tfidf_rows.append(row)

# nnz là số phần tử khác 0 trong ma trận TF-IDF sparse.
nnz = sum(len(row) for row in tfidf_rows)
total_elements = N * V
# Sparsity là tỷ lệ phần tử bằng 0; density là tỷ lệ phần tử khác 0.
sparsity = 1 - nnz / total_elements
density = nnz / total_elements

assert N > 0 and V > 0
assert 0 <= density <= 1 and 0 <= sparsity <= 1
print('Count/TF-IDF sparse representation created successfully.')

Count/TF-IDF sparse representation created successfully.


## 4. kiểm tra kích thước ma trận và sparsity


In [4]:
print(f'Number of documents (N) = {N:,}')
print(f'Vocabulary size (V) = {V:,}')
print(f'Matrix shape = ({N:,}, {V:,})')
print(f'nnz(X) = {nnz:,}')
print(f'Total matrix elements = {total_elements:,}')
print(f'Sparsity = {sparsity:.6f} ({sparsity * 100:.4f}%)')
print(f'Density = {density:.6f} ({density * 100:.4f}%)')

Number of documents (N) = 30,000
Vocabulary size (V) = 193,837
Matrix shape = (30,000, 193,837)
nnz(X) = 5,104,560
Total matrix elements = 5,815,110,000
Sparsity = 0.999122 (99.9122%)
Density = 0.000878 (0.0878%)


### Trả lời:
Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có
chiều (V)?

Một document chỉ chứa một phần rất nhỏ vocabulary nên chỉ có một số ít phần tử khác 0
nhưng tất cả document phải dùng cùng một vocabulary chung để các cột có cùng ý nghĩa giữa các document
nên mỗi vector vẫn có V chiều; phần lớn các chiều không xuất hiện trong document đó và được lưu bằng 0

## 5. Inspect vocabulary


In [5]:
# Hàm phụ để in bảng kết quả theo format dễ đọc.
def print_table(title, rows, headers):
    print('\n' + title)
    print(' | '.join(headers))
    print('-' * 80)
    for row in rows:
        print(' | '.join(str(value) for value in row))

# Xếp term theo document frequency giảm dần và lấy 20 term đầu.
top_df = sorted(((term, df[term], df[term] / N) for term in vocabulary), key=lambda x: (-x[1], x[0]))[:20]
print_table('Top 20 terms by document frequency', [(t, f'{freq:,}', f'{ratio:.6f}') for t, freq, ratio in top_df], ['term', 'df', 'df/N'])

# Xếp term theo IDF giảm dần: các term hiếm sẽ đứng đầu.
top_idf = sorted(((term, idf[term], df[term]) for term in vocabulary), key=lambda x: (-x[1], x[0]))[:20]
print_table('Top 20 terms by IDF', [(t, f'{value:.6f}', freq) for t, value, freq in top_idf], ['term', 'idf', 'df'])

# Chọn document số 0 để inspect các term có TF-IDF cao nhất.
chosen_doc_index = 0
chosen_doc_tfidf = sorted(tfidf_rows[chosen_doc_index].items(), key=lambda x: (-x[1], x[0]))[:20]
print(f'\nSelected document index = {chosen_doc_index}')
print('URL =', documents[chosen_doc_index]['url'])
print('Text preview =', texts[chosen_doc_index][:300].replace('\n', ' '))
print_table('Top 20 terms by TF-IDF in selected document', [(t, f'{value:.8f}', df[t], f'{idf[t]:.6f}') for t, value in chosen_doc_tfidf], ['term', 'tfidf', 'df', 'idf'])


Top 20 terms by document frequency
term | df | df/N
--------------------------------------------------------------------------------
the | 27,893 | 0.929767
and | 27,423 | 0.914100
to | 26,689 | 0.889633
of | 26,031 | 0.867700
a | 25,905 | 0.863500
in | 25,224 | 0.840800
for | 23,651 | 0.788367
is | 22,739 | 0.757967
with | 21,405 | 0.713500
on | 20,262 | 0.675400
that | 18,370 | 0.612333
this | 17,840 | 0.594667
are | 17,594 | 0.586467
it | 17,168 | 0.572267
s | 16,959 | 0.565300
as | 16,467 | 0.548900
at | 16,347 | 0.544900
from | 16,316 | 0.543867
be | 16,153 | 0.538433
you | 16,094 | 0.536467

Top 20 terms by IDF
term | idf | df
--------------------------------------------------------------------------------
00000 | 10.308953 | 1
000000 | 10.308953 | 1
00000000 | 10.308953 | 1
0000000000000965 | 10.308953 | 1
00000001 | 10.308953 | 1
00000048 | 10.308953 | 1
000002 | 10.308953 | 1
000004b926f1 | 10.308953 | 1
00000781 | 10.308953 | 1
0000085054 | 10.308953 | 1
00000xxx | 10.308953

### Câu hỏi
Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?

Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?

- Term xuất hiện nhiều lần trong corpuss không nhất thiết phải có TF-IDF cao vì có thể term đó là các từ phổ biến thì IDF thấp nên TF-IDF chưa chắc cao



- Term có IDF cao thì xuất hiện trong rất ít document nên không phải tất cả đều xuất hiện nhiều trong mọi document vì vậy chưa chắc TF-IDF cao trong mọi document


# Part F — Experiment 2: Preprocessing Ablation



In [6]:
import gc

# Stopword list cố định để thí nghiệm B có thể tái lập.
STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from',
    'has', 'have', 'he', 'her', 'his', 'i', 'in', 'is', 'it', 'its',
    'of', 'on', 'or', 'that', 'the', 'their', 'there', 'they', 'this',
    'to', 'was', 'we', 'were', 'will', 'with', 'you', 'your'
}

# Pipeline A: chỉ lowercase và tokenization.
def minimal_tokens(text):
    return TOKEN_RE.findall(text.lower())

# Pipeline B: chuẩn hóa punctuation, tokenize và tùy chọn bỏ stopword.
def normalized_words(text, remove_stopwords=True):
    cleaned = re.sub(r'[^\w\s]', ' ', text.lower())
    words = TOKEN_RE.findall(cleaned)
    if remove_stopwords:
        words = [word for word in words if word not in STOPWORDS]
    return words

# Pipeline C: giữ word token và bổ sung prefix/suffix subword units.
def extended_tokens(text):
    words = normalized_words(text, remove_stopwords=False)
    subwords = []
    for word in words:
        if len(word) >= 3:
            subwords.append('sub_prefix:' + word[:3])
            subwords.append('sub_suffix:' + word[-3:])
    return words + subwords

# Lưu function trong dictionary để chạy cùng một phép đo cho A, B, C.
PIPELINES = {
    'A Minimal': minimal_tokens,
    'B Normalized': normalized_words,
    'C Extended': extended_tokens,
}

QUERIES = ['classification', 'machine learning', 'healthcare', 'natural language processing', 'data model']
print('Queries used for every pipeline:', QUERIES)

Queries used for every pipeline: ['classification', 'machine learning', 'healthcare', 'natural language processing', 'data model']


## Hàm fit sparse và search


In [7]:
# Fit một pipeline: tạo count rows, vocabulary, IDF, document norms và nnz.
def fit_sparse(tokenizer):
    # Streaming build: avoid keeping a second list containing every token string.
    # Streaming giúp không phải giữ thêm list khổng lồ của toàn bộ token strings.
    count_rows = []
    token_lengths = []
    document_frequency = Counter()
    vocabulary_terms = set()
    # Mỗi document được tokenize và chuyển thành Counter sparse.
    for text in texts:
        tokens = tokenizer(text)
        counts = Counter(tokens)
        count_rows.append(counts)
        token_lengths.append(len(tokens))
        vocabulary_terms.update(counts.keys())
        document_frequency.update(counts.keys())
    # Mỗi term có một column index ổn định trong vocabulary.
    vocab = {term: index for index, term in enumerate(sorted(vocabulary_terms))}
    idf_values = {term: math.log(N / document_frequency[term]) for term in vocab}
    # Lưu norm trước để cosine search không phải tính lại norm document.
    document_norms = []
    nnz_value = 0
    for length, counts in zip(token_lengths, count_rows):
        squared_norm = sum((count / length * idf_values[term]) ** 2 for term, count in counts.items() if length and idf_values[term] != 0.0)
        document_norms.append(math.sqrt(squared_norm))
        nnz_value += sum(idf_values[term] != 0.0 for term in counts)
    return token_lengths, vocab, idf_values, count_rows, document_norms, document_frequency, nnz_value

# Chuyển query sang cùng không gian TF-IDF với document index.
def query_vector(query, tokenizer, vocab, idf_values):
    tokens = tokenizer(query)
    counts = Counter(tokens)
    total = len(tokens)
    vector = {term: count / total * idf_values[term] for term, count in counts.items() if total and term in vocab and idf_values[term] != 0.0}
    return vector, tokens

# Cosine similarity cho hai vector sparse dạng dictionary.
def sparse_cosine(x, y, norm_x=None, norm_y=None):
    if norm_x is None:
        norm_x = math.sqrt(sum(value * value for value in x.values()))
    if norm_y is None:
        norm_y = math.sqrt(sum(value * value for value in y.values()))
    if not norm_x or not norm_y:
        return 0.0
    smaller, larger = (x, y) if len(x) <= len(y) else (y, x)
    dot = sum(value * larger.get(term, 0.0) for term, value in smaller.items())
    return dot / (norm_x * norm_y)

# Tính cosine với từng document, sắp xếp và lấy Top-K.
def search_top1(query, tokenizer, vocab, idf_values, count_rows, document_norms, token_lengths, k=5):
    q_vector, query_tokens = query_vector(query, tokenizer, vocab, idf_values)
    # Query không có term hợp lệ sẽ có norm bằng 0.
    q_norm = math.sqrt(sum(value * value for value in q_vector.values()))
    scored = []
    for index, counts in enumerate(count_rows):
        length = token_lengths[index]
        # Chỉ lặp qua query terms để tính dot product sparse.
        dot = sum(q_value * counts.get(term, 0) / length * idf_values[term] for term, q_value in q_vector.items()) if length else 0.0
        d_norm = document_norms[index]
        # Cosine = dot product / tích hai norm; tránh chia cho 0.
        score = dot / (q_norm * d_norm) if q_norm and d_norm else 0.0
        scored.append((score, index))
    return sorted(scored, key=lambda item: (-item[0], item[1]))[:k], query_tokens

##  bảng so sánh


In [8]:
# Lưu metric của ba pipeline để in bảng so sánh cuối cell.
ablation_results = []

for pipeline_name, tokenizer in PIPELINES.items():
    token_lengths, vocab, idf_values, count_rows, document_norms, document_frequency, pipeline_nnz = fit_sparse(tokenizer)
    pipeline_v = len(vocab)
    pipeline_avg_tokens = sum(token_lengths) / N
    # Tính sparsity trên ma trận TF-IDF của pipeline hiện tại.
    pipeline_sparsity = 1 - pipeline_nnz / (N * pipeline_v)
    all_query_tokens = []
    top1_scores = []
    for query in QUERIES:
        ranked, query_tokens = search_top1(query, tokenizer, vocab, idf_values, count_rows, document_norms, token_lengths)
        all_query_tokens.extend(query_tokens)
        top1_scores.append(ranked[0][0] if ranked else 0.0)
    oov_count = sum(token not in vocab for token in all_query_tokens)
    oov_rate = oov_count / len(all_query_tokens) if all_query_tokens else 0.0
    mean_top1 = sum(top1_scores) / len(top1_scores)
    row = {
        'pipeline': pipeline_name,
        'vocabulary_size': pipeline_v,
        'average_tokens_per_document': pipeline_avg_tokens,
        'matrix_sparsity': pipeline_sparsity,
        'oov_rate': oov_rate,
        'mean_top1_cosine_proxy': mean_top1,
    }
    ablation_results.append(row)
    print('\n' + pipeline_name)
    for metric, value in row.items():
        if metric != 'pipeline':
            print(f'  {metric}: {value:.6f}' if isinstance(value, float) else f'  {metric}: {value:,}')
    del token_lengths, vocab, idf_values, count_rows, document_norms, document_frequency
    gc.collect()

print('\nComparison table')
print('Pipeline | Vocabulary | Avg tokens/doc | Sparsity | OOV rate | Mean Top-1 cosine proxy')
print('-' * 105)
for row in ablation_results:
    print(f"{row['pipeline']} | {row['vocabulary_size']:,} | {row['average_tokens_per_document']:.2f} | {row['matrix_sparsity']:.6f} | {row['oov_rate']:.4f} | {row['mean_top1_cosine_proxy']:.6f}")


A Minimal
  vocabulary_size: 193,837
  average_tokens_per_document: 369.696367
  matrix_sparsity: 0.999122
  oov_rate: 0.000000
  mean_top1_cosine_proxy: 0.437542

B Normalized
  vocabulary_size: 193,800
  average_tokens_per_document: 247.391267
  matrix_sparsity: 0.999222
  oov_rate: 0.000000
  mean_top1_cosine_proxy: 0.439663

C Extended
  vocabulary_size: 231,691
  average_tokens_per_document: 951.835167
  matrix_sparsity: 0.998301
  oov_rate: 0.000000
  mean_top1_cosine_proxy: 0.437746

Comparison table
Pipeline | Vocabulary | Avg tokens/doc | Sparsity | OOV rate | Mean Top-1 cosine proxy
---------------------------------------------------------------------------------------------------------
A Minimal | 193,837 | 369.70 | 0.999122 | 0.0000 | 0.437542
B Normalized | 193,800 | 247.39 | 0.999222 | 0.0000 | 0.439663
C Extended | 231,691 | 951.84 | 0.998301 | 0.0000 | 0.437746


##9.5. Câu hỏi phân tích
Trả lời bằng kết quả thực nghiệm, không chỉ bằng lý thuyết:
1. Lowercasing làm thay đổi vocabulary như thế nào?
2. Stopword removal có luôn cải thiện representation không?
3. Việc loại punctuation có thể làm mất thông tin gì?
4. Pipeline nào tạo ra sparse matrix nhất?
5. Pipeline nào cho search tốt nhất?
6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?

Dùng bảng kết quả vừa in để trả lời, không khẳng định preprocessing càng nhiều càng tốt:

1. **Lowercasing làm thay đổi vocabulary thế nào?** So sánh A với B; do A đã lowercase nên thay đổi chính của B đến từ punctuation và stopword handling.
2. **Stopword removal có luôn cải thiện representation không?** Không nhất thiết. Nó có thể giảm vocabulary và số token, nhưng có thể làm mất tín hiệu ngữ nghĩa hoặc thay đổi search.
3. **Loại punctuation có thể mất thông tin gì?** Dấu câu có thể biểu thị cảm xúc, phủ định, cấu trúc câu, số thập phân, mã phiên bản hoặc ký hiệu kỹ thuật.
4. **Pipeline nào sparse nhất?** Chọn pipeline có matrix_sparsity lớn nhất trong bảng.
5. **Pipeline nào search tốt nhất?** Chỉ được kết luận chắc chắn sau khi có relevance labels và P@5/Recall@5/MRR. Nếu chỉ dùng kết quả hiện tại, hãy gọi pipeline có mean_top1_cosine_proxy cao nhất là pipeline có proxy score cao nhất.
6. **Search tốt hơn có đồng nghĩa vocabulary nhỏ hơn không?** Không. Vocabulary size và chất lượng search là hai thuộc tính khác nhau; hãy đối chiếu trực tiếp hai cột trong bảng.

# Part G — Application: Build a Document Search Engine



In [9]:
# sử dụng lại pipeline B đã xây dựng ở bài F
search_token_lengths, search_vocab, search_idf, search_count_rows, search_doc_norms, search_df, search_nnz = fit_sparse(normalized_words)
SEARCH_QUERIES = [
    'medical image classification',
    'transformer language model',
    'deep learning healthcare',
    'natural language processing',
]

print(f'Search index documents = {N:,}')
print(f'Search index vocabulary = {len(search_vocab):,}')
print(f'Search index sparsity = {1 - search_nnz / (N * len(search_vocab)):.6f}')

Search index documents = 30,000
Search index vocabulary = 193,800
Search index sparsity = 0.999222


In [10]:
# hàm in kết quả tìm được
def print_search_results(query, ranked, query_tokens):
    print('\n' + '=' * 120)
    print('QUERY:', query)
    known_tokens = [token for token in query_tokens if token in search_vocab]
    unknown_tokens = [token for token in query_tokens if token not in search_vocab]
    print('Query tokens:', query_tokens)
    print('Known tokens:', known_tokens if known_tokens else 'none')
    print('OOV tokens:', unknown_tokens if unknown_tokens else 'none')
    if not known_tokens:
        print('WARNING: query has no token in the vocabulary; all similarities are 0.')
    print('\nRank | Document ID | Similarity | Document preview | URL')
    print('-' * 120)
    for rank, (score, row_index) in enumerate(ranked, start=1):
        preview = ' '.join(texts[row_index].split())[:200]
        url = documents[row_index].get('url', '')
        print(f'{rank:>4} | {row_index:>11} | {score:>10.6f} | {preview} | {url}')


search_output_rows = []
for query in SEARCH_QUERIES:
    ranked, query_tokens = search_top1(
        query,
        normalized_words,
        search_vocab,
        search_idf,
        search_count_rows,
        search_doc_norms,
        search_token_lengths,
        k=5,
    )
    print_search_results(query, ranked, query_tokens)
    for rank, (score, row_index) in enumerate(ranked, start=1):
        search_output_rows.append({
            'query': query,
            'rank': rank,
            'document_id': row_index,
            'similarity': score,
            'url': documents[row_index].get('url', ''),
            'preview': ' '.join(texts[row_index].split())[:200],
        })

print(f'\nSaved in-memory search rows: {len(search_output_rows)}')


QUERY: medical image classification
Query tokens: ['medical', 'image', 'classification']
Known tokens: ['medical', 'image', 'classification']
OOV tokens: none

Rank | Document ID | Similarity | Document preview | URL
------------------------------------------------------------------------------------------------------------------------
   1 |       18971 |   0.444917 | The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning construction projects and who want to build in an environmentally responsible manner. The enviro | http://glt.rts.fi/etusivu/rts-ymparistoluokitus/rts-glt-environmental-classification-information-in-english/
   2 |        8527 |   0.367218 | History of maize classification. How races used in classification. Geographical distribution. Existing races of maize in Mexico. | https://books.google.rs/books?id=tXxQAAAAMAAJ&amp;hl=sr&amp;source=gbs_book_other_versions_r&amp;cad=4
   3 |       19908 |   0.234543 | Download Leag

## Observation guide for Part G

Sau khi chạy, hãy đọc preview và URL của từng Top-5 để tự nhận xét:

1. Query nào trả về các document có nội dung phù hợp nhất?
2. Query nào có similarity cao hoặc thấp bất thường?
3. Có document nào chứa đúng nhiều query terms nhưng vẫn không đứng đầu không? Có thể do TF-IDF giảm trọng số các term quá phổ biến.
4. Nếu query có OOV token, hệ thống vẫn chạy; token đó bị bỏ qua khi tạo query vector.
5. Nếu toàn bộ query là OOV, cosine similarity của mọi document bằng 0 và kết quả chỉ là tie-break theo document ID.

Similarity cao chỉ cho biết vector gần query hơn; chưa đủ để kết luận document thực sự relevant. Muốn đánh giá search quality cần thêm relevance labels ở Part H.

# Part H — Evaluation


## 1. Tạo evaluation set và điền relevance labels

Notebook dùng 5 query. Document ID chính là chỉ số dòng của document trong corpus, được in ở Part G. Bạn cần tự đọc preview/URL của Top-5 rồi điền các ID relevant vào dictionary dưới đây.

Ví dụ: RELEVANCE_LABELS['medical image classification'] = {12, 45, 78}. Không điền document chỉ vì nó đứng Top-5; chỉ điền khi bạn kiểm tra nội dung và thấy relevant.

In [13]:
EVAL_QUERIES = SEARCH_QUERIES + ['data model']

# TODO: tự điền document IDs relevant sau khi đọc kết quả Part G.
# Các set rỗng là placeholder, không phải kết quả đánh giá.
RELEVANCE_LABELS = {
    'medical image classification': set(),
    'transformer language model': set(),
    'deep learning healthcare': set(),
    'natural language processing': set(),
    'data model': set(),
}

assert set(EVAL_QUERIES) == set(RELEVANCE_LABELS)
assert all(isinstance(ids, set) for ids in RELEVANCE_LABELS.values())
print('Evaluation queries:', len(EVAL_QUERIES))
print('Labels filled:', sum(bool(ids) for ids in RELEVANCE_LABELS.values()), '/', len(EVAL_QUERIES))

Evaluation queries: 5
Labels filled: 0 / 5


## 2. Precision@5, Recall@5 và Reciprocal Rank

Với mỗi query, hệ thống lấy Top-5 document từ cùng search index của Part G:

- P@5 = số document relevant trong Top-5 / 5.
- Recall@5 = số document relevant trong Top-5 / tổng số document relevant đã gán nhãn.
- RR = 1 / rank của document relevant đầu tiên; nếu không có relevant document trong Top-5 thì RR = 0.
- MRR là trung bình RR trên các query có labels.

Nếu labels còn rỗng, code chỉ báo pending và không tự tạo metric giả.

In [14]:
def evaluate_one_query(query, relevant_ids, k=5):
    ranked, query_tokens = search_top1(
        query,
        normalized_words,
        search_vocab,
        search_idf,
        search_count_rows,
        search_doc_norms,
        search_token_lengths,
        k=k,
    )
    retrieved_ids = [row_index for score, row_index in ranked]
    relevant_ids = set(relevant_ids)
    hits = [doc_id for doc_id in retrieved_ids if doc_id in relevant_ids]
    precision_at_k = len(hits) / k
    recall_at_k = len(hits) / len(relevant_ids) if relevant_ids else None
    reciprocal_rank = next((1 / rank for rank, doc_id in enumerate(retrieved_ids, start=1) if doc_id in relevant_ids), 0.0)
    assert 0.0 <= precision_at_k <= 1.0
    assert 0.0 <= reciprocal_rank <= 1.0
    if recall_at_k is not None:
        assert 0.0 <= recall_at_k <= 1.0
    return {
        'query': query,
        'retrieved_ids': retrieved_ids,
        'relevant_ids': sorted(relevant_ids),
        'hits': hits,
        'precision_at_5': precision_at_k,
        'recall_at_5': recall_at_k,
        'reciprocal_rank': reciprocal_rank,
    }


evaluation_rows = []
for query in EVAL_QUERIES:
    result = evaluate_one_query(query, RELEVANCE_LABELS[query], k=5)
    evaluation_rows.append(result)
    if result['relevant_ids']:
        print(f"{query}: P@5={result['precision_at_5']:.4f}, Recall@5={result['recall_at_5']:.4f}, RR={result['reciprocal_rank']:.4f}, hits={result['hits']}")
    else:
        print(f'{query}: pending — chưa có relevance labels')

labeled_rows = [row for row in evaluation_rows if row['relevant_ids']]
if labeled_rows:
    mean_precision = sum(row['precision_at_5'] for row in labeled_rows) / len(labeled_rows)
    mean_recall = sum(row['recall_at_5'] for row in labeled_rows) / len(labeled_rows)
    mrr = sum(row['reciprocal_rank'] for row in labeled_rows) / len(labeled_rows)
    print('\nMean metrics over labeled queries')
    print(f'Mean P@5 = {mean_precision:.4f}')
    print(f'Mean Recall@5 = {mean_recall:.4f}')
    print(f'MRR = {mrr:.4f}')
else:
    mean_precision = mean_recall = mrr = None
    print('\nEvaluation is pending: hãy điền RELEVANCE_LABELS rồi chạy lại cell này.')

medical image classification: pending — chưa có relevance labels
transformer language model: pending — chưa có relevance labels
deep learning healthcare: pending — chưa có relevance labels
natural language processing: pending — chưa có relevance labels
data model: pending — chưa có relevance labels

Evaluation is pending: hãy điền RELEVANCE_LABELS rồi chạy lại cell này.


## 3. Interpretation

Sau khi điền labels, báo cáo ba metric chính: mean P@5, mean Recall@5 và MRR. P@5 cao nghĩa là Top-5 chứa nhiều document relevant; Recall@5 cao nghĩa là Top-5 tìm được nhiều trong số các document relevant đã biết; MRR cao nghĩa là document relevant đầu tiên thường xuất hiện ở rank cao.

Nếu một query có Recall@5 thấp, không được kết luận ngay rằng preprocessing hoặc TF-IDF sai. Hãy kiểm tra thêm: relevance labels có đầy đủ không, query có OOV không, document relevant có thật sự nằm trong corpus không, và từ khóa query có bị stopword handling loại bỏ không.

# Code walkthrough — Giải thích chi tiết

Notebook phải chạy từ trên xuống dưới vì các cell sau sử dụng biến do các cell trước tạo ra.

## 1. Tìm và đọc dataset

Path giúp thao tác đường dẫn trên local và Google Colab. candidate_paths chứa các vị trí thường gặp của dataset. next sẽ lấy path đầu tiên thật sự tồn tại. Nếu đang ở Colab và chưa tìm thấy file, notebook clone repository GitHub vào /content/NLP rồi dùng rglob tìm dataset trong các thư mục con.

File dataset là JSONL, nghĩa là mỗi dòng là một JSON object riêng. json.loads(line) chuyển dòng thành Python dictionary. item.get('text', '') lấy trường text; nếu JSON hỏng thì invalid_count tăng và dòng đó bị bỏ qua. Document không có token nào được tính vào empty_count. Document hợp lệ được lưu cùng text và url.

Sau cell này: texts chứa nội dung document, N là số document hợp lệ, token_lists là danh sách token của từng document, còn token_lengths dùng để tính độ dài trung bình, nhỏ nhất và lớn nhất.

## 2. Tokenizer

TOKEN_RE dùng regular expression (?u)\\b\\w+\\b. \\w+ lấy một hoặc nhiều ký tự word, còn \\b đánh dấu boundary của word. Hàm tokenize chuyển text thành string, lowercase rồi trả về danh sách token.

Ví dụ: text Hello, World! sẽ trở thành danh sách ['hello', 'world']. Lowercase làm cho Hello và hello được xem là cùng một term.

## 3. Part D — Count, TF, IDF và TF-IDF

Vocabulary được tạo bằng set comprehension để lấy mỗi term một lần. sorted làm thứ tự cột ổn định. enumerate đánh số cột từ 0. V là số term khác nhau.

count_rows dùng Counter cho từng document. Nếu document có cat cat fish thì Counter là {'cat': 2, 'fish': 1}. Chỉ term xuất hiện mới được lưu, nên đây là sparse representation.

df là document frequency. Với từng Counter, row.keys() chỉ chứa term xuất hiện trong document đó. df.update(row.keys()) cộng một lần cho mỗi document, không cộng theo số lần term lặp trong document.

idf[term] = log(N / df[term]) áp dụng đúng công thức đề bài. Term xuất hiện trong mọi document có IDF bằng log(1), tức bằng 0.

Trong vòng lặp TF-IDF, count / length là TF. Nhân TF với idf tạo TF-IDF. Chỉ value khác 0 mới được đưa vào row. Vì vậy mỗi dictionary trong tfidf_rows là một hàng sparse của ma trận X.

nnz là tổng số phần tử khác 0. Ma trận lý thuyết có N nhân V phần tử. density = nnz / (N nhân V), còn sparsity = 1 - density. Hai assert cuối kiểm tra kích thước và tỷ lệ có hợp lệ hay không.

## 4. Inspect vocabulary

Hàm print_table chỉ dùng để in dữ liệu. top_df tạo các tuple gồm term, df và df/N, sau đó sắp xếp df giảm dần và lấy 20 phần tử đầu. top_idf làm tương tự nhưng sắp xếp theo IDF giảm dần.

chosen_doc_index chọn document cần kiểm tra. tfidf_rows[chosen_doc_index].items() lấy các cặp term và TF-IDF của document đó. Sắp xếp value giảm dần sẽ cho 20 term có trọng số cao nhất trong document.

DF cao nghĩa là term phổ biến toàn corpus. IDF cao nghĩa là term hiếm. TF-IDF cao phụ thuộc cả độ thường xuyên trong document đang xét và độ hiếm trên toàn corpus, nên ba danh sách có thể khác nhau.

## 5. Part F — Preprocessing ablation

Pipeline A chỉ lowercase và tokenize. Pipeline B thay punctuation bằng space, tokenize và loại stopword. Pipeline C giữ word token rồi thêm prefix và suffix subword cho các word dài ít nhất 3 ký tự.

PIPELINES là dictionary chứa tên pipeline và function tokenizer. Nhờ vậy cùng một phép đo có thể chạy lần lượt cho A, B và C.

fit_sparse là đoạn khó nhất của Part F. Với từng text, tokenizer tạo token list và Counter tạo count row. vocabulary_terms gom các term khác nhau, còn document_frequency đếm số document chứa từng term. Sau khi quét corpus, code tính IDF, document norm và nnz. Cách làm này không tạo ma trận dense N nhân V.

document norm là căn bậc hai của tổng bình phương các giá trị TF-IDF. Norm được tính trước để lúc search không phải tính lại cho từng query.

## 6. Query vector và cosine search

query_vector tokenize query, đếm query term, bỏ các term không có trong vocabulary và tính TF-IDF cho các term còn lại. Term không có trong vocabulary được gọi là OOV.

search_top1 tính q_norm là độ dài vector query. Với mỗi document, counts.get(term, 0) lấy số lần query term xuất hiện; nếu không có thì lấy 0. Biểu thức count / length nhân IDF là TF-IDF của term trong document. Tổng các tích query-value và document-value là dot product.

Cosine similarity bằng dot chia cho q_norm nhân document norm. Nếu một norm bằng 0 thì score bằng 0 để tránh lỗi chia cho 0. sorted sắp xếp score giảm dần, dùng document index làm tie-break, rồi lấy Top-K.

## 7. Part G — Search engine

Part G fit lại Pipeline B để tạo search index. Mỗi query được chuyển thành vector TF-IDF, so sánh với toàn bộ document và in Top-5. known_tokens là token có trong vocabulary; unknown_tokens là OOV.

Nếu toàn bộ query là OOV thì query vector bằng 0, mọi similarity bằng 0 và kết quả chỉ được sắp xếp theo document ID. preview được chuẩn hóa thành một dòng và giới hạn 200 ký tự. search_output_rows lưu query, rank, document ID, similarity, URL và preview để có thể dùng tiếp cho evaluation.

## 8. Part H — Evaluation

RELEVANCE_LABELS là dictionary do người làm bài tự điền sau khi đọc preview và URL. Mỗi value là set Document ID relevant. Không nên gán nhãn chỉ vì document đứng Top-5.

evaluate_one_query lấy Top-5, tạo retrieved_ids và tìm hits, tức các document vừa được retrieve vừa có trong relevance labels. Precision@5 = số hits chia 5. Recall@5 = số hits chia tổng số relevant đã gán nhãn. Reciprocal Rank là 1 chia rank của hit đầu tiên; nếu không có hit thì bằng 0.

labeled_rows loại những query chưa có labels. Mean P@5, Mean Recall@5 và MRR chỉ tính trên labeled rows. Vì vậy khi labels còn rỗng, notebook báo pending thay vì tự bịa ra kết quả.

Để hoàn thành Part H: chạy Part G, đọc năm bảng Top-5, điền Document ID vào RELEVANCE_LABELS, rồi chạy lại cell tạo labels và cell evaluation.